In [13]:
#Load and split documents

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

content= """Pega is a powerful low-code enterprise software platform designed to streamline workflow automation, customer relationship management, and artificial intelligence-driven decision-making. 

### Where It Is Used
* Large organizations and enterprises.
* **Financial sectors** like banking and insurance for claims processing and customer onboarding.
* **Regulated industries** including healthcare, telecommunications, and government agencies.

### Benefits
* **Fast Development:** The drag-and-drop, low-code interface allows teams to build and deploy applications quickly without writing extensive code from scratch.
* **Automation:** It combines Business Process Management (BPM) and Robotic Process Automation (RPA) to handle repetitive tasks and reduce human error.
* **Customer Focus:** Real-time AI decisioning helps businesses personalize customer service interactions and accelerate response times.

### Disadvantages
* **High Cost:** The software requires expensive enterprise licensing fees and a substantial financial investment.
* **Steep Learning Curve:** Configuring advanced rules and managing the platform effectively requires specialized developer training.
* **Vendor Lock-in:** Because it is a proprietary system rather than open-source, migrating away from the platform can be difficult and costly."""


In [14]:
with open ('pega_intro.txt','w',encoding='utf-8') as file:
    file.write(content)

loader = TextLoader('pega_intro.txt', encoding='utf-8')
documents = loader.load()

# Splits documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap = 20,
    length_function=len,
    separators=["\n\n","\n",""," "]
)

splits = text_splitter.split_documents(documents)

print(f"Split into {len(splits)} chunks")
for i, split in enumerate(splits):
    print(f"Chunk: {i}, {split.page_content}")

Split into 20 chunks
Chunk: 0, Pega is a powerful low-code enterprise software platform designed to streamline workflow automation,
Chunk: 1, workflow automation, customer relationship management, and artificial intelligence-driven decision-m
Chunk: 2, ce-driven decision-making.
Chunk: 3, ### Where It Is Used
* Large organizations and enterprises.
Chunk: 4, * **Financial sectors** like banking and insurance for claims processing and customer onboarding.
Chunk: 5, * **Regulated industries** including healthcare, telecommunications, and government agencies.
Chunk: 6, ### Benefits
Chunk: 7, * **Fast Development:** The drag-and-drop, low-code interface allows teams to build and deploy appl
Chunk: 8, uild and deploy applications quickly without writing extensive code from scratch.
Chunk: 9, * **Automation:** It combines Business Process Management (BPM) and Robotic Process Automation (RPA
Chunk: 10, cess Automation (RPA) to handle repetitive tasks and reduce human error.
Chunk: 11, * **Cust

In [15]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS

embedding_function = HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") 

print("Creating Embedding and Indexing in FAISS")

db = FAISS.from_documents(
    documents=documents,
    embedding=embedding_function
)
print("Vector DB Successfully created")
print(f"Stored {len(splits)} vectors")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2316.00it/s]


Creating Embedding and Indexing in FAISS
Vector DB Successfully created
Stored 20 vectors


In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
model_id = 'Qwen/Qwen2-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

pipe = pipeline(
    "text-generation",
    model,
    tokenizer=tokenizer,
    max_new_tokens = 128,
    temperature = 0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM Setup Completed")

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 131.88it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Setup Completed


In [17]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
def format_qwen(input_dict):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant which answers questions based on "
                "the provided context. Do not use any external knowledge or make "
                "up any content. If the answer is not in the context, just reply: "
                "'I don't know the answer to this query based on your provided context.' "
                "Always answer your query in the following format: "
                "'Answer': 'Your answer'"
            )
        },
        {
            "role": "user",
            "content": f"""
Context:

{input_dict['context']}

Question:

{input_dict['question']}
"""
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

def generate_with_qwen(formatted_prompt):
    response = llm.invoke(formatted_prompt)

    generated = response.split(formatted_prompt)[-1].strip()

    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[-1].strip()

    return generated

llm_with_format = (
    RunnableLambda(format_qwen)
    | RunnableLambda(generate_with_qwen)
    | StrOutputParser()
)

print("LLM Setup Complete")

LLM Setup Complete


In [18]:
import threading
import logging
from flask import Flask, request, jsonify
from flask_cors import CORS
import time

In [19]:
def create_qwen_prompt (user_input:str, history:list=None):
    if history is None:
        history=[]

    messages = [{
    "role": "system",
    "content": "You are a helpful AI Assistant"
}]
    messages.extend(history)
    messages.append({"role":"user","content":user_input})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def create_qwen_response(response: str, prompt: str) -> str:
    if prompt in response:
        return response.replace(prompt,"").strip()
    if "<|im_start|>assistant" in response:
        return response.split("<|im_start|>assistant")[-1].strip()
    return response.strip()

In [20]:
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

retriever = db.as_retriever(search_kwargs={"k":2})

rag_chain = (
    {
        "context": retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
        "question": RunnablePassthrough()
    } | llm_with_format
)

In [21]:
#Initializing Flask
app = Flask(__name__)
CORS(app)
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

In [22]:
@app.route('/rag-query', methods=['POST'])
def rag_query():
    try:
        data = request.json
        answer = rag_chain.invoke(data['question'])
        return jsonify({"answer": answer})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

In [23]:
def run_flask():
    print("Flask server started...")
    app.run(port=5001, use_reloader=False)

t= threading.Thread(target=run_flask)
t.daemon=True
t.start()
print("Waiting for server to be ready...")
time.sleep(3)

Flask server started...
Waiting for server to be ready...
 * Serving Flask app '__main__'
 * Debug mode: off


In [24]:
#Testing piepline
import requests
print("Testing Flask RAG App")
user_query = str(input("Enter your Query"))
try:
    resp = requests.post(
        "http://127.0.0.1:5001/rag-query", json=({"question": user_query})
    )
    if resp.status_code == 200:
        print(f"Question: {user_query}")
        print(f"Answer: {resp.json()['answer']}")
    else:
        print(f"Error: {resp.text}, {resp.status_code}")
except Exception as e:
    print(f"Exception: {e}")
time.sleep(1)

Testing Flask RAG App


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question: What are the benefits of using PEGA
Answer: The benefits of using PEGA include fast development, automation, and customer focus.
